In [62]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

## Basic Content-Based Movie Recommender

In the first part of this project, I build a **basic content-based recommender** that suggests movies similar to a given title using only movie metadata, not user histories. This is a classic starting point for recommendation systems and works well as a simple, item-centric “more like this” engine.

The idea is to represent each movie by its **genres** and then measure how similar two movies are in this feature space. I use `TfidfVectorizer` to convert the cleaned genre strings (`genres_clean`) into numerical vectors, where each dimension corresponds to a genre token and each value reflects its importance in that movie relative to the whole catalog. With these TF–IDF vectors, I compute **cosine similarity** between movies; the cosine of the angle between two vectors gives a similarity score between 0 and 1, independent of their absolute length. 

To generate recommendations, I:

- Build a reverse index from movie titles to dataframe indices so I can quickly locate any requested movie.  
- Take the TF–IDF vector for the chosen movie and compute cosine similarity against all other movies in the dataset.  
- Sort the candidate movies by similarity (and later also by quality signals such as average rating and number of ratings) and return the top‑N as recommendations.

This first version is deliberately simple and **non-personalized**: it only answers “Which movies are most similar to X?” based on content features. It forms the foundation on which I later add more data science steps such as filtering by popularity, reranking with rating information, enforcing minimum genre overlap, adding diversity, and building user-like profiles from multiple input movies.

In [3]:
# Load ratings
ratings = pd.read_csv("ml-32m/ratings.csv")

# Group by movie to get the average rating and total number of ratings
movie_stats = ratings.groupby("movieId")["rating"].agg(["mean", "count"]).reset_index()
movie_stats.rename(columns={"mean": "avg_rating", "count": "num_ratings"}, inplace=True)

# EXCLUDE movies with fewer than 50 ratings
popular_movies_stats = movie_stats[movie_stats["num_ratings"] >= 50]

display(popular_movies_stats.head())


,movieId,avg_rating,num_ratings
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


In [31]:
# Load movies
movies = pd.read_csv("ml-32m/movies.csv")
print(movies.columns)

# Merge movies with our popular_movies_stats
# We use an 'inner' join, so any movie NOT in popular_movies_stats gets dropped instantly!
movies = movies.merge(popular_movies_stats, on="movieId", how="inner")

# Clean the genres text
movies["genres"] = movies["genres"].replace("(no genres listed)", "")
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False).str.lower() #remove the | bar

# Reset index because we dropped a lot of rows during the merge
movies = movies.reset_index(drop=True)

display(movies[["title", "genres_clean", "avg_rating", "num_ratings"]].head())


Index(['movieId', 'title', 'genres'], dtype='object')


,title,genres_clean,avg_rating,num_ratings
0,Toy Story (1995),adventure animation children comedy fantasy,3.897438,68997
1,Jumanji (1995),adventure children fantasy,3.275758,28904
2,Grumpier Old Men (1995),comedy romance,3.139447,13134
3,Waiting to Exhale (1995),comedy drama romance,2.845331,2806
4,Father of the Bride Part II (1995),comedy,3.059602,13154


In [7]:
#Read data before cleaning
print("Head:\n", movies.head())
print("\Shape:\n", movies.shape)
print("\Count:\n", movies[['genres_clean']].value_counts())
print("\nDtypes:\n", movies.dtypes)
print("\nMissing Values:\n", movies.isnull().sum())
print("\nDuplicates:", movies.duplicated().sum())
print("\nDescribe:\n", movies.describe())
print(movies.columns)


Head:
    movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  avg_rating  num_ratings  \
0  Adventure|Animation|Children|Comedy|Fantasy    3.897438        68997   
1                   Adventure|Children|Fantasy    3.275758        28904   
2                               Comedy|Romance    3.139447        13134   
3                         Comedy|Drama|Romance    2.845331         2806   
4                                       Comedy    3.059602        13154   

                                  genres_clean  
0  adventure animation children comedy fantasy  
1                   adventure children fantasy  
2                               comedy romance  
3                         comedy drama romanc

In [75]:
# Initialize and apply TF-IDF
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres_clean"])

movies.to_csv("movies_merged.csv", index=False) #save csv file to be used later

print(f"New TF-IDF Matrix Shape (Filtered): {tfidf_matrix.shape}")

New TF-IDF Matrix Shape (Filtered): (16034, 21)


In [43]:
# Reverse mapping
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

def get_recommendations(title, top_n=10, min_shared_genres=1):
    """ Grabs top n most similar movies to the one passed """
    
    if title not in indices:
        return f"Error: '{title}' not found. It might have fewer than 50 ratings."
    
    idx = indices[title]
    target_genres = set(movies.loc[idx, "genres_clean"].split())

    # Filter to movies that share at least `min_shared_genres` with target
    mask = movies["genres_clean"].apply(
        lambda g: len(target_genres & set(g.split())) >= min_shared_genres
    )
    mask.iloc[idx] = False            # still drop the movie itself
    candidate_idx = movies[mask].index

    # Similarity only on this subset
    sim_scores_array = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix[candidate_idx]
    ).flatten()

    recommended = movies.loc[candidate_idx].copy()
    recommended["similarity_score"] = sim_scores_array
    recommended = recommended[recommended.index != idx] #drop input

    recommended = recommended[recommended["num_ratings"] >= 20] #filter movies with low number of ratings
    recommended = rerank(recommended, alpha=0.8) #rerank the movies using ratings threshold


    return recommended[["title","genres","similarity_score","avg_rating","num_ratings"]].head(top_n)

In [45]:
def rerank(df, alpha=0.8):
    """ Movies below a no. of ratings threshold are not included in recommendation """
    
    # normalize columns to 0–1
    df = df.copy()
    df["sim_norm"] = (df["similarity_score"] - df["similarity_score"].min()) / (
        df["similarity_score"].max() - df["similarity_score"].min()
    )
    df["rating_norm"] = (df["avg_rating"] - df["avg_rating"].min()) / (
        df["avg_rating"].max() - df["avg_rating"].min()
    )
    df["final_score"] = alpha * df["sim_norm"] + (1 - alpha) * df["rating_norm"]
    return df.sort_values("final_score", ascending=False)

In [47]:
test_movie = "Toy Story (1995)"
display(get_recommendations(test_movie))


,title,genres,similarity_score,avg_rating,num_ratings
15954,Puss in Boots: The Last Wish (2022),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.921983,1301
15488,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.911948,4486
4385,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.837442,46036
13918,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.816381,8278
2786,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.812043,32683
3585,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.650783,11437
15622,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.638158,1216
15338,Onward (2020),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.596420,1732
12996,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.973825,3.909975,21483
12362,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.334826,893


In [49]:
# Save the cleaned dataframe
movies.to_pickle("single_movie_recommender.pkl")

# Save the TF-IDF matrix
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")


['tfidf_matrix.pkl']

## Making the Content-Based Recommender More “User-Specific”

So far, my content-based recommender answers the question “Given one movie X, what are the most similar movies?”. This is useful, but it is still item-centric rather than user-centric. In this step, I extend the system in two simple ways to make it feel more personalized while staying in the content-based framework, without collaborative filtering or user IDs. [web:219][web:85]

### User profile from multiple liked movies

Real users usually like several movies, not just one. A standard content-based trick is to build a user profile vector by aggregating the feature vectors of movies they like and then recommending items close to that profile. [web:219][web:223]

The idea with TF–IDF features is:

- The user supplies a list of liked titles, for example `["Toy Story (1995)", "Jumanji (1995)", "Aladdin (1992)"]`.  
- I look up the row indices for these titles and extract their TF–IDF vectors from `tfidf_matrix`.  
- I compute the average of these vectors to form a single “taste” vector representing the user’s preferences across all chosen movies.  
- I compute cosine similarity between this averaged profile and every movie in the catalog.  
- I remove the input movies themselves, optionally require some genre overlap, filter out movies with very few ratings, and then apply the same reranking step as before (combining similarity and average rating into `final_score`). [web:219][web:220]

This yields a new function, `get_recommendations_from_list(titles, ...)`, that returns movies similar to the set of input titles, not just a single title. It behaves like a lightweight user-specific recommender where the user’s “profile” is defined by the movies they manually choose.

### Genre-based top-N browsing

As a second, simpler extension, I add a function that lets a user specify one or more genres, for example `["comedy", "romance"]`, and returns the top‑N movies that match those genres, ranked by average rating and number of ratings. Genre-based recommendation is a common pattern in movie systems and works well as a browsing tool for users who want the best movies in a particular genre mix. [web:225][web:227]

The logic is:

- The user picks a set of genres.  
- I filter `movies` to rows whose `genres_clean` contains all selected genres.  
- I keep only movies above a minimum `num_ratings` threshold for reliability.  
- I sort by `avg_rating` (and break ties with `num_ratings`) and return the top‑N titles. [web:227][web:229]

This is not personalized in the collaborative-filtering sense, but it adds a useful exploration path that complements the “similar to X” and “similar to a set of movies” functions.

### Why this improves the project

These additions strengthen the project in several ways:

- They show a progression from single-item similarity to a simple notion of user taste, using vector aggregation. [web:219][web:221]  
- They demonstrate how to reuse the same TF–IDF, cosine similarity, and reranking pipeline across different interaction patterns: a single seed movie, multiple seed movies, and pure genre filters. [web:85][web:220]  
- They enrich the user experience, letting someone ask for recommendations based on several favorites at once or browse highly rated movies within chosen genres, without needing a full user-account system.

In the next code cells, I implement `get_recommendations_from_list(titles, top_n, ...)` for profile-based recommendations from multiple liked movies and `get_top_movies_by_genres(selected_genres, top_n, ...)` for top‑N movies by genre.

In [64]:
def get_recommendations_from_list(titles, top_n=10, min_shared_genres=1, alpha=0.8):
    valid_titles = [t for t in titles if t in indices]
    if not valid_titles:
        return "Error: none of the input titles were found."

    input_idx = [indices[t] for t in valid_titles]

    # Build a user profile by averaging the TF-IDF vectors
    user_profile = np.asarray(tfidf_matrix[input_idx].mean(axis=0))
        
    # Similarity of profile vs all movies
    sim_scores_array = cosine_similarity(user_profile, tfidf_matrix).flatten()

    recommended = movies.copy()
    recommended["similarity_score"] = sim_scores_array

    # Remove input movies
    recommended = recommended[~recommended.index.isin(input_idx)].copy()

    # Optional genre filter: use union of all input genres
    target_genres = set()
    for idx in input_idx:
        target_genres |= set(movies.loc[idx, "genres_clean"].split())

    mask = recommended["genres_clean"].apply(
        lambda g: len(target_genres & set(g.split())) >= min_shared_genres
    )
    recommended = recommended[mask]

    # Filter low-support movies
    recommended = recommended[recommended["num_ratings"] >= 20]

    # Rerank
    recommended = rerank(recommended, alpha=alpha)

    return recommended[[
        "title", "genres", "similarity_score",
        "avg_rating", "num_ratings", "final_score"
    ]].head(top_n)

In [66]:
get_recommendations_from_list(
    ["Toy Story (1995)", "Jumanji (1995)", "Aladdin (1992)"],
    top_n=10
)

,title,genres,similarity_score,avg_rating,num_ratings,final_score
15954,Puss in Boots: The Last Wish (2022),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.921983,1301,0.977353
15488,Soul (2020),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.911948,4486,0.976770
4385,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.837442,46036,0.972447
13918,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.816381,8278,0.971225
2786,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.812043,32683,0.970973
3585,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.650783,11437,0.961615
15622,Luca (2021),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.638158,1216,0.960882
15338,Onward (2020),Adventure|Animation|Children|Comedy|Fantasy,0.954885,3.596420,1732,0.958460
9850,Ponyo (Gake no ue no Ponyo) (2008),Adventure|Animation|Children|Fantasy,0.935500,3.847689,5236,0.955944
13767,Kubo and the Two Strings (2016),Adventure|Animation|Children|Fantasy,0.935500,3.847373,3312,0.955926


In [68]:
def get_top_movies_by_genres(selected_genres, top_n=10, min_votes=20):
    selected_genres = set(g.lower() for g in selected_genres)

    filtered = movies[movies["genres_clean"].apply(
        lambda g: selected_genres.issubset(set(g.split()))
    )].copy()

    filtered = filtered[filtered["num_ratings"] >= min_votes]

    return filtered.sort_values(
        by=["avg_rating", "num_ratings"],
        ascending=[False, False]
    )[["title", "genres", "avg_rating", "num_ratings"]].head(top_n)

In [70]:
get_top_movies_by_genres(["comedy", "romance"], top_n=10)

,title,genres,avg_rating,num_ratings
12629,Pride and Prejudice (1980),Comedy|Drama|Romance,4.231481,54
2048,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.150374,29819
800,"Philadelphia Story, The (1940)",Comedy|Drama|Romance,4.131498,7445
1065,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957
12689,Operation 'Y' & Other Shurik's Adventures (1965),Comedy|Crime|Romance,4.117547,587
807,It Happened One Night (1934),Comedy|Romance,4.114604,5074
11608,Paperman (2012),Animation|Comedy|Romance,4.106174,3725
2947,City Lights (1931),Comedy|Drama|Romance,4.097747,3995
4464,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.082169,43459
853,His Girl Friday (1940),Comedy|Romance,4.079538,4897
